In [58]:
import sys, os, json, uuid
from datetime import datetime

sys.path.insert(0, os.path.abspath('..'))

from config.settings import settings
from services.knowledge_base import get_knowledge_base_prompt
from services.memory_service import get_session, format_session_for_prompt

DATABASE_ID = 'degreefyd_online_lms'

def print_section(title, content, max_chars=4000):
    bar = '=' * 80
    print(f'\n{bar}\n{title}\n{bar}')
    s = str(content)
    print(s[:max_chars] + ('\n...[TRUNCATED]' if len(s) > max_chars else ''))

def inspect_session_history(session_id):
    session = get_session(session_id)
    formatted = format_session_for_prompt(session_id)
    return session, formatted

def reconstruct_prompt(user_query, session_history, database_id=DATABASE_ID):
    knowledge_base = get_knowledge_base_prompt(database_id=database_id)
    now = datetime.now()
    date_str = now.strftime('%A, %d %B %Y')
    time_str = now.strftime('%I:%M %p')
    prompt = (
        knowledge_base + '\n' + session_history +
        '\n\n### CURRENT SYSTEM CONTEXT:\n'
        f'- Today Date (IST): {date_str}\n'
        f'- System Time (IST): {time_str}\n\n'
        '### STRICT INSTRUCTIONS:\n'
        'Must Follow These -\n'
        'Use small well-structured CTEs to simplify logic.\n'
        'Prefer window functions (LEAD, LAG, ROW_NUMBER, RANK) wherever applicable.\n\n'
        '1. MANDATORY: Use CONVERSATION HISTORY for follow-up context.\n'
        '2. MANDATORY: JOIN counsellors table for staff names.\n'
        '3. MANDATORY: COUNT(DISTINCT student_id) for admissions.\n'
        '4. MANDATORY: Apply date filters ONLY if explicitly specified.\n'
        '5. MANDATORY: No LIMIT unless user says top N.\n'
        '6. MANDATORY: Limit 5 for queries returning more than 50 rows.\n'
        '7. TIMEZONE: Rolling windows use NOW() - INTERVAL; IST day-boundary subtract 5h30m offset.\n'
        '8. Year queries use EXTRACT(YEAR FROM CURRENT_DATE). Never hardcode year.\n'
        '9. NEVER ILIKE on IDs. Always join by name.\n'
        '10. Output ONLY raw SQL.\n\n'
        f'User Question: "{user_query}"\nSQL:'
    )
    return prompt

print(f'SESSION_MAX_TURNS : {settings.SESSION_MAX_TURNS}')
print(f'Knowledge base dir: {settings.KNOWLEDGE_BASE_DIR}')
print('Ready - no server or login needed.')

SESSION_MAX_TURNS : 5
Knowledge base dir: C:\Users\mohit\OneDrive\Desktop\FRESH START\DataWhisper\knowledge_base
Ready - no server or login needed.


## Step 1 - Inspect Knowledge Base

In [59]:
kb = get_knowledge_base_prompt(database_id=DATABASE_ID)
print_section('Knowledge Base (first 3000 chars)', kb[:3000])
print(f'\nTotal KB length: {len(kb)} chars')


Knowledge Base (first 3000 chars)
Online KB 

# TEXT-TO-SQL KNOWLEDGE BASE
# READ SECTION 6 (CRITICAL RULES) BEFORE WRITING ANY SQL.

---

# SECTION 1 — TABLE SCHEMAS

---

## TABLE: students
One row per student/lead. Primary funnel record.

| Column | Type | Notes |
| :--- | :--- | :--- |
| student_id | VARCHAR PK | Format STD-XXXX. Always COUNT(DISTINCT student_id). |
| student_name | VARCHAR | Filter with ILIKE. |
| student_email | VARCHAR | Unique. |
| student_phone | VARCHAR | Unique. |
| current_student_status | VARCHAR | **Only source of truth for funnel stage.** |
| current_student_ni_sub_status | VARCHAR | Sub-reason for NI. Only populated when `current_student_status = 'NotInterested'`. Valid values: 'Not Enquired', 'Multiple Attempts made', 'Reason not shared', 'Course Not Available', 'Already Enrolled_NP', 'Only_Regular course', 'Invalid number / Wrong Number', 'First call Not Interested', 'Next Year', 'Budget issue', 'Not Eligible', 'Already Enrolled_Partner', 'Language B

## Step 2 - Load a real session from DB

Paste a `session_id` / `chat_id` from your app (visible in browser network tab or from a recent query response).

In [60]:
SESSION_ID = 'e7d628e0-8442-4537-9ab4-81aefc39b00d'

session_data, formatted_history = inspect_session_history(SESSION_ID)

print(f'Turns in session: {len(session_data.get("turns", []))}')
print_section('Formatted Session History (exactly as sent to LLM)', formatted_history)

Fetching session e7d628e0-8442-4537-9ab4-81aefc39b00d from DB
Fetching session e7d628e0-8442-4537-9ab4-81aefc39b00d from DB
Formatted session e7d628e0-8442-4537-9ab4-81aefc39b00d for prompt with 1 turns (including feedback)
Turns in session: 1

Formatted Session History (exactly as sent to LLM)

### CONVERSATION HISTORY (use this for follow-up questions):

User: hello
SQL: SELECT 'Hello! How can I assist you with your data today?' AS message;
Answer: 



## Step 3 - Inspect each turn in detail

In [61]:
turns = session_data.get('turns', [])
print(f'Total turns stored: {len(turns)}\n')
for i, turn in enumerate(turns):
    print(f'--- Turn {i+1} ---')
    print(f'  Query   : {turn.get("user_query", "")}')
    sql = turn.get('generated_sql', '') or ''
    print(f'  SQL     : {sql[:300] + "..." if len(sql) > 300 else sql}')
    ans = turn.get('answer', '') or ''
    print(f'  Answer  : {ans[:200] + "..." if len(ans) > 200 else ans}')
    print(f'  Verdict : {turn.get("query_verdict", "(no verdict yet)")}')
    print(f'  Reason  : {turn.get("failure_reason", "")}')
    print()

Total turns stored: 1

--- Turn 1 ---
  Query   : hello
  SQL     : SELECT 'Hello! How can I assist you with your data today?' AS message;
  Answer  : 
  Verdict : (no verdict yet)
  Reason  : 



## Step 4 - Reconstruct FULL prompt that was sent for the follow-up query

Set `FOLLOWUP_QUERY` to Q2 (the one that failed) to see exactly what the LLM received.

In [62]:
FOLLOWUP_QUERY = 'this data in source and campign level'

full_prompt = reconstruct_prompt(FOLLOWUP_QUERY, formatted_history)
print_section('FULL PROMPT SENT TO LLM FOR FOLLOW-UP QUERY', full_prompt, max_chars=8000)


FULL PROMPT SENT TO LLM FOR FOLLOW-UP QUERY
Online KB 

# TEXT-TO-SQL KNOWLEDGE BASE
# READ SECTION 6 (CRITICAL RULES) BEFORE WRITING ANY SQL.

---

# SECTION 1 — TABLE SCHEMAS

---

## TABLE: students
One row per student/lead. Primary funnel record.

| Column | Type | Notes |
| :--- | :--- | :--- |
| student_id | VARCHAR PK | Format STD-XXXX. Always COUNT(DISTINCT student_id). |
| student_name | VARCHAR | Filter with ILIKE. |
| student_email | VARCHAR | Unique. |
| student_phone | VARCHAR | Unique. |
| current_student_status | VARCHAR | **Only source of truth for funnel stage.** |
| current_student_ni_sub_status | VARCHAR | Sub-reason for NI. Only populated when `current_student_status = 'NotInterested'`. Valid values: 'Not Enquired', 'Multiple Attempts made', 'Reason not shared', 'Course Not Available', 'Already Enrolled_NP', 'Only_Regular course', 'Invalid number / Wrong Number', 'First call Not Interested', 'Next Year', 'Budget issue', 'Not Eligible', 'Already Enrolled_Partner', '

## Step 5 - Test generating SQL directly (no API, calls Gemini)

This calls `generate_sql` exactly as the server would, so you see real output.

In [63]:
from services.llm_service import generate_sql

Q1 = 'what unusual did u experience in yesterday lead flow as compared to its pervious day'
Q2 = 'this data in source and campign level'

print('Generating SQL for Q1 (no session history)...')
sql_q1, thoughts_q1, usage_q1 = generate_sql(
    user_query=Q1,
    session_history='',
    database_id=DATABASE_ID
)
print(f'SQL Q1:\n{sql_q1}')
print(f'Tokens: {usage_q1}')

Generating SQL for Q1 (no session history)...
SQL Q1:
WITH daily_leads AS (
    SELECT 
        DATE_TRUNC('day', created_at AT TIME ZONE 'Asia/Kolkata') AS lead_date,
        COUNT(DISTINCT student_id) AS total_leads,
        COUNT(DISTINCT CASE WHEN source ILIKE '%Facebook%' THEN student_id END) AS fb_leads,
        COUNT(DISTINCT CASE WHEN source ILIKE '%Google%' THEN student_id END) AS google_leads,
        COUNT(DISTINCT CASE WHEN current_student_status = 'NotInterested' THEN student_id END) AS ni_leads
    FROM students
    WHERE created_at >= (CURRENT_DATE - INTERVAL '2 days' - INTERVAL '5 hours 30 minutes')
      AND created_at < (CURRENT_DATE - INTERVAL '5 hours 30 minutes')
    GROUP BY 1
),
comparison AS (
    SELECT 
        lead_date,
        total_leads,
        LAG(total_leads) OVER (ORDER BY lead_date) AS prev_day_total,
        fb_leads,
        LAG(fb_leads) OVER (ORDER BY lead_date) AS prev_day_fb,
        google_leads,
        LAG(google_leads) OVER (ORDER BY lead_d

In [64]:
print('Generating SQL for Q2 WITHOUT session history (simulate broken context)...')
sql_q2_no_ctx, _, usage = generate_sql(
    user_query=Q2,
    session_history='',
    database_id=DATABASE_ID
)
print(f'SQL Q2 (no context):\n{sql_q2_no_ctx}')

Generating SQL for Q2 WITHOUT session history (simulate broken context)...
SQL Q2 (no context):
SELECT
    s.source,
    sla.utm_campaign,
    COUNT(DISTINCT s.student_id) AS total_leads,
    COUNT(DISTINCT CASE WHEN s.current_student_status = 'NotInterested' THEN s.student_id END) AS ni_leads,
    COUNT(DISTINCT CASE WHEN csj.course_status = 'Application' THEN csj.student_id END) AS applications,
    COUNT(DISTINCT CASE WHEN csj.course_status IN ('Admission', 'Enrolled') AND (csj.course_status = 'Enrolled' OR csj.fee_type NOT IN ('partial paid', 'Partially Paid', 'Partial Done')) THEN csj.student_id END) AS admissions
FROM students s
LEFT JOIN student_lead_activities sla ON s.student_id = sla.student_id
LEFT JOIN course_status_journeys csj ON s.student_id = csj.student_id
GROUP BY s.source, sla.utm_campaign
ORDER BY total_leads DESC;


In [65]:
simulated_history = (
    '\n### CONVERSATION HISTORY (use this for follow-up questions):\n\n'
    f'User: {Q1}\n'
    f'SQL: {sql_q1}\n'
    'Answer: Lead flow showed unusual drop in volume yesterday vs previous day.\n\n'
)

print('Generating SQL for Q2 WITH simulated Q1 context...')
sql_q2_with_ctx, _, usage = generate_sql(
    user_query=Q2,
    session_history=simulated_history,
    database_id=DATABASE_ID
)
print(f'SQL Q2 (with context):\n{sql_q2_with_ctx}')

Generating SQL for Q2 WITH simulated Q1 context...
SQL Q2 (with context):
SELECT 
    s.source,
    sla.utm_campaign,
    COUNT(DISTINCT s.student_id) AS total_leads,
    COUNT(DISTINCT CASE WHEN s.current_student_status = 'NotInterested' THEN s.student_id END) AS ni_leads,
    COUNT(DISTINCT CASE WHEN csj.course_status = 'Application' THEN csj.student_id END) AS applications,
    COUNT(DISTINCT CASE WHEN csj.course_status IN ('Admission', 'Enrolled') AND (csj.course_status = 'Enrolled' OR csj.fee_type NOT IN ('partial paid', 'Partially Paid', 'Partial Done')) THEN csj.student_id END) AS conversions
FROM students s
LEFT JOIN student_lead_activities sla ON s.student_id = sla.student_id
LEFT JOIN course_status_journeys csj ON s.student_id = csj.student_id
GROUP BY s.source, sla.utm_campaign
ORDER BY total_leads DESC;


## Step 6 - Compare: what changed between Q2 with vs without context?

Look at the two SQLs above:
- **Without context**: Does Q2 produce a generic or wrong query?
- **With context**: Does Q2 correctly inherit date filter + metric from Q1 and add source/campaign GROUP BY?

If it still fails with context, the problem is either:
1. The Q1 SQL stored in session is too long / complex for the model to parse
2. The follow-up phrasing (`this data`) is too vague and needs a prompt instruction fix
3. The few-shot examples don't cover this aggregation-change pattern

In [66]:
print('=== SIDE BY SIDE ===')
print(f'Q2 WITHOUT context:\n{sql_q2_no_ctx}\n')
print('-' * 60)
print(f'Q2 WITH context:\n{sql_q2_with_ctx}')

=== SIDE BY SIDE ===
Q2 WITHOUT context:
SELECT
    s.source,
    sla.utm_campaign,
    COUNT(DISTINCT s.student_id) AS total_leads,
    COUNT(DISTINCT CASE WHEN s.current_student_status = 'NotInterested' THEN s.student_id END) AS ni_leads,
    COUNT(DISTINCT CASE WHEN csj.course_status = 'Application' THEN csj.student_id END) AS applications,
    COUNT(DISTINCT CASE WHEN csj.course_status IN ('Admission', 'Enrolled') AND (csj.course_status = 'Enrolled' OR csj.fee_type NOT IN ('partial paid', 'Partially Paid', 'Partial Done')) THEN csj.student_id END) AS admissions
FROM students s
LEFT JOIN student_lead_activities sla ON s.student_id = sla.student_id
LEFT JOIN course_status_journeys csj ON s.student_id = csj.student_id
GROUP BY s.source, sla.utm_campaign
ORDER BY total_leads DESC;

------------------------------------------------------------
Q2 WITH context:
SELECT 
    s.source,
    sla.utm_campaign,
    COUNT(DISTINCT s.student_id) AS total_leads,
    COUNT(DISTINCT CASE WHEN s.curre

In [ ]:
from query_rewriter import rewrite_to_standalone, maybe_rewrite

print("query_rewriter loaded.")

query_rewriter loaded.


In [ ]:
# Test rewrite_to_standalone with your actual Q1/Q2
print(f"Previous query : {Q1}")
print(f"Follow-up query: {Q2}")
print()

# New signature: takes list of previous queries
previous_queries = [Q1]
rewritten = rewrite_to_standalone(previous_queries, Q2)
print(f"Rewritten query: {rewritten}")

Previous query : what unusual did u experience in yesterday lead flow as compared to its pervious day
Follow-up query: this data in source and campign level

Rewritten query: What unusual patterns did you experience in lead flow yesterday compared to the previous day, broken down by source and campaign level?


In [69]:
# Full pipeline: Q2 → rewrite → generate_sql
# This is what your router would do if you integrate query_rewriter.py

session_turns = [{"user_query": Q1, "generated_sql": sql_q1, "answer": ""}]

final_query, was_rewritten = maybe_rewrite(Q2, session_turns)

print(f"Original  : {Q2}")
print(f"Rewritten : {final_query}")
print(f"Was rewritten: {was_rewritten}")
print()

print("Generating SQL from rewritten query...")
sql_rewritten, _, usage = generate_sql(
    user_query=final_query,
    session_history='',       # no need to pass history — query is now standalone
    database_id=DATABASE_ID
)
print(f"Tokens: {usage}")
print(f"\nSQL from rewritten query:\n{sql_rewritten}")

# Check if date filter is present
has_date = any(k in sql_rewritten.lower() for k in ['current_date', 'interval', 'now()'])
print(f"\nHas date filter: {has_date} {'OK' if has_date else 'STILL MISSING'}")

Original  : this data in source and campign level
Rewritten : What unusual patterns did you experience in lead flow yesterday compared to the previous day, broken down by source and campaign level?
Was rewritten: True

Generating SQL from rewritten query...
Tokens: {'input_tokens': 36286, 'output_tokens': 525}

SQL from rewritten query:
WITH yesterday_data AS (
    SELECT 
        s.source,
        COALESCE(sla.utm_campaign, 'Direct/Organic') AS campaign,
        COUNT(DISTINCT s.student_id) AS lead_count
    FROM students s
    LEFT JOIN student_lead_activities sla ON s.student_id = sla.student_id
    WHERE s.created_at >= CURRENT_DATE - INTERVAL '1 day' - INTERVAL '5 hours 30 minutes'
      AND s.created_at < CURRENT_DATE - INTERVAL '5 hours 30 minutes'
    GROUP BY 1, 2
),
prev_day_data AS (
    SELECT 
        s.source,
        COALESCE(sla.utm_campaign, 'Direct/Organic') AS campaign,
        COUNT(DISTINCT s.student_id) AS lead_count
    FROM students s
    LEFT JOIN student_lead_

In [ ]:
import time

# Each test: (prev_query, followup_query, expected_to_have_date_filter)
PERF_CASES = [
    (
        "what unusual did u experience in yesterday lead flow as compared to its pervious day",
        "this data in source and campign level",
        True,
    ),
    (
        "how many leads came in yesterday vs previous day",
        "same but show by counsellor",
        True,
    ),
    (
        "show me top counsellors by admissions this month",
        "above data for last month",
        True,
    ),
    (
        "how many leads are hot right now",
        "break this down by source",
        True,  # no date filter in original either
    ),
    (
        "what is the conversion rate this week",
        "same data by campaign level",
        True,
    ),
    (
        "show admissions in april",
        "those students by university",
        True,
    ),
    (
        "admissions and icc today",
        "and forms",
        True,
    ),
]

print(f"{'#':<3} {'Follow-up Query':<45} {'Rewritten Query':<65} {'Date Filter':<12} {'Time(s)'}")
print("-" * 145)

results = []
for i, (prev_q, followup_q, expect_date) in enumerate(PERF_CASES):
    t0 = time.time()
    # New signature: pass list of previous queries
    rewritten = rewrite_to_standalone([prev_q], followup_q)
    elapsed = round(time.time() - t0, 2)

    # Check if the rewritten query still carries temporal context
    has_date_word = any(w in rewritten.lower() for w in [
        "yesterday", "today", "last month", "this month", "this week",
        "april", "last week", "previous day", "today", "now"
    ])

    status = "OK" if (has_date_word == expect_date) else "MISS"
    results.append(status)

    print(f"{i+1:<3} {followup_q:<45} {rewritten[:63]:<65} {str(has_date_word):<12} {elapsed}")

print()
passed = results.count("OK")
print(f"Score: {passed}/{len(PERF_CASES)} passed")

#   Follow-up Query                               Rewritten Query                                                   Date Filter  Time(s)
-------------------------------------------------------------------------------------------------------------------------------------------------
1   this data in source and campign level         What unusual patterns did you experience in lead flow yesterday   True         2.05
2   same but show by counsellor                   How many leads came in yesterday vs the previous day, broken do   True         0.71
3   above data for last month                     show me top counsellors by admissions for last month              True         0.88
4   break this down by source                     How many hot leads are there right now, broken down by source?    True         1.04
5   same data by campaign level                   what is the conversion rate this week grouped by campaign level   True         1.18
6   those students by university               

In [ ]:
## Step 7 - Test new continuation pattern with multiple queries

# Test the pattern: admission today -> and forms -> and icc done -> and ni
print("Testing continuation pattern with multiple queries...\n")

queries_sequence = [
    "admission today",
    "and forms",
    "and icc done",
    "and ni",
]

session_turns = []
print(f"{'#':<3} {'Query':<40} {'Rewritten':<60} {'Action'}")
print("-" * 110)

for i, query in enumerate(queries_sequence):
    final_query, was_rewritten = maybe_rewrite(query, session_turns)
    
    action = "STANDALONE" if not was_rewritten else "MERGED"
    print(f"{i+1:<3} {query:<40} {final_query[:58]:<60} {action}")
    
    # Add to session for next iteration
    session_turns.append({"user_query": final_query, "generated_sql": "", "answer": ""})

print("\n✓ Continuation pattern test complete")

In [ ]:
## Step 8 - Test correction pattern (fixing previous wrong query)

print("Testing correction pattern...\n")

# Scenario: User makes a query, then realizes it's incomplete and corrects it
session_turns = [
    {"user_query": "admission today", "generated_sql": "", "answer": ""}
]

correction_query = "admission and forms today"
final_query, was_rewritten = maybe_rewrite(correction_query, session_turns)

print(f"Previous query: {session_turns[0]['user_query']}")
print(f"Correction query: {correction_query}")
print(f"Rewritten query: {final_query}")
print(f"Was rewritten: {was_rewritten}")

if was_rewritten:
    print("\n✓ Correction pattern: LLM intelligently merged the correction")
else:
    print("\n✗ Correction pattern: Query treated as standalone")

In [ ]:
## Step 9 - Test with 5 previous queries (last 5 from session)

print("Testing with last 5 queries from session history...\n")

# Simulate a session with 5 previous queries
session_turns = [
    {"user_query": "leads yesterday", "generated_sql": "", "answer": ""},
    {"user_query": "admissions today", "generated_sql": "", "answer": ""},
    {"user_query": "forms this week", "generated_sql": "", "answer": ""},
    {"user_query": "icc done last month", "generated_sql": "", "answer": ""},
    {"user_query": "ni by source", "generated_sql": "", "answer": ""},
]

# Now a follow-up that might refer to any of the previous queries
followup_query = "same but for yesterday"
final_query, was_rewritten = maybe_rewrite(followup_query, session_turns)

print(f"Session has {len(session_turns)} previous queries")
print(f"Follow-up query: {followup_query}")
print(f"Rewritten query: {final_query}")
print(f"Was rewritten: {was_rewritten}")

if was_rewritten:
    print("\n✓ LLM used last 5 queries to determine context")
else:
    print("\n✗ Query treated as standalone")

In [ ]:
# Full pipeline perf test: rewrite → generate_sql → check SQL has date filter
# Costs tokens per case — runs all 6 cases

print(f"{'#':<3} {'Follow-up':<45} {'Rewritten':<50} {'SQL Date Filter':<16} {'Time(s)'}")
print("-" * 125)

sql_results = []
for i, (prev_q, followup_q, expect_date) in enumerate(PERF_CASES):
    t0 = time.time()

    # New signature: pass list of previous queries
    rewritten = rewrite_to_standalone([prev_q], followup_q)
    sql_out, _, usage = generate_sql(
        user_query=rewritten,
        session_history='',
        database_id=DATABASE_ID
    )
    elapsed = round(time.time() - t0, 2)

    has_date = any(k in sql_out.lower() for k in ['current_date', 'interval', 'now()', 'extract'])
    status = "OK" if (has_date == expect_date) else "MISS"
    sql_results.append({"status": status, "followup": followup_q, "rewritten": rewritten, "sql": sql_out, "has_date": has_date})

    print(f"{i+1:<3} {followup_q:<45} {rewritten[:48]:<50} {str(has_date):<16} {elapsed}")

print()
passed = sum(1 for r in sql_results if r["status"] == "OK")
print(f"Score: {passed}/{len(sql_results)} passed")
print()

# Print failures in detail
for r in sql_results:
    if r["status"] == "MISS":
        print(f"MISS: {r['followup']}")
        print(f"  Rewritten : {r['rewritten']}")
        print(f"  SQL snippet: {r['sql'][:300]}")
        print()

#   Follow-up                                     Rewritten                                          SQL Date Filter  Time(s)
-----------------------------------------------------------------------------------------------------------------------------
1   this data in source and campign level         What unusual patterns did you experience in lead   True             3.98
2   same but show by counsellor                   How many leads came in yesterday vs the previous   True             3.96
3   above data for last month                     show me top counsellors by admissions for last m   True             3.15
4   break this down by source                     How many hot leads are there right now, broken d   False            3.05
5   same data by campaign level                   what is the conversion rate this week grouped by   True             4.06
6   those students by university                  show admissions in april grouped by university     True             3.14

Score: 6/